# 🐍 Team Slytherine — Colab Training & Inference Pipeline
### Amazon ML Challenge 2026: Business Entity Resolution

This notebook connects to your GitHub repository, pulls the latest code, and runs the entire training and inference pipeline using Colab GPU & High RAM.

**Hardware Recommendation:** Runtime → Change runtime type → **A100 GPU** or **L4 GPU** + **High-RAM**.

## Step 1: Mount Google Drive
Make sure your dataset folder is in Google Drive under `MyDrive/amazonML/student_resource/dataset/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/amazonML'
DATA_DIR = f'{DRIVE_DIR}/student_resource/dataset'
OUT_DIR = f'{DRIVE_DIR}/Amazon-ML-challenge/output'
REP_DIR = f'{DRIVE_DIR}/Amazon-ML-challenge/reports'

assert os.path.exists(f'{DATA_DIR}/train/train_source1.tsv'), f'❌ Dataset not found at {DATA_DIR}/train/'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)
print('✅ Google Drive mounted and dataset verified!')

## Step 2: Clone or Pull Latest Code from GitHub
Whenever Antigravity fixes an error or pushes improvements, just re-run this cell to pull the latest changes!

In [ ]:
%cd /content
REPO_URL = 'https://github.com/navyadeshmukh/Amazon-ML-challenge.git'

if not os.path.exists('/content/Amazon-ML-challenge'):
    !git clone $REPO_URL
else:
    %cd /content/Amazon-ML-challenge
    !git pull

%cd /content/Amazon-ML-challenge
!git log -1 --oneline

## Step 3: Install Dependencies & Check GPU

In [ ]:
%cd /content/Amazon-ML-challenge
!pip install -q -r code/requirements.txt
!pip install -q -r code/requirements-embeddings.txt

import torch, psutil
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU (CPU)'
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
ram = psutil.virtual_memory().total / (1024**3)
print(f'✅ GPU: {gpu} ({vram:.1f} GB VRAM)')
print(f'✅ System RAM: {ram:.1f} GB')

## Step 4: Fast 1-Minute Sanity Run (Recommended First)
Verifies data loading, blocking, features, training, and output generation before starting the long run.

In [ ]:
%cd /content/Amazon-ML-challenge/code/business_entity_resolution
!python -u run_pipeline.py \
  --data-root $DATA_DIR \
  --out-root $OUT_DIR \
  --report-dir $REP_DIR \
  --quick

## Step 5: Full Production Training & Inference Run
Runs the full dataset across all 12.5M records with live unbuffered logging (`-u`).

In [ ]:
# === RECOMMENDED: High-Recall Baseline (Fast, 99.8% Recall, 0% OOM risk) ===
%cd /content/Amazon-ML-challenge/code/business_entity_resolution
!python -u run_pipeline.py \
  --data-root $DATA_DIR \
  --out-root $OUT_DIR \
  --report-dir $REP_DIR \
  --loco

# === OPTION B: With Pretrained Embeddings on GPU (Requires High-RAM runtime) ===
# !python -u run_pipeline.py \
#   --data-root $DATA_DIR \
#   --out-root $OUT_DIR \
#   --report-dir $REP_DIR \
#   --embedder bgem3 \
#   --device cuda \
#   --loco

## Step 6: Validate Submission Files
Runs the official competition validator to ensure 100% compliance.

In [ ]:
%cd /content/drive/MyDrive/amazonML/student_resource
!python utils/validate_submission.py \
  --matching $OUT_DIR/matching_results.tsv \
  --candidate $OUT_DIR/candidate_pairs.tsv \
  --test-dir dataset/test

## Step 7: Build Final Submission Zip

In [ ]:
%cd /content/Amazon-ML-challenge/code/business_entity_resolution
!python src/make_submission_zip.py --team Slytherine --solution-root /content/Amazon-ML-challenge

# Copy zip to Google Drive for easy download
!cp /content/Amazon-ML-challenge/Slytherine_submission.zip $DRIVE_DIR/
print(f'✅ Slytherine_submission.zip saved to Google Drive: {DRIVE_DIR}/Slytherine_submission.zip')